<a href="https://colab.research.google.com/github/NICKSTER0506/IoT-Intrusion-Detection-Deployment-Gap/blob/main/Comparison_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [8]:
dataset = pd.read_csv('/content/drive/MyDrive/IoT-IDS-Shared/clustered_dataset.csv')

In [9]:
dataset.describe().T

,count,mean,std,min,25%,50%,75%,max
2318.4402090840676,44999.0,2400.084634,48.239944,2199.611246,2367.560809,2400.045211,2432.608640,2596.367862
0.9095661384526199,44999.0,0.008632,3.019591,-13.191679,-2.025444,-0.003796,2.035190,12.358836
-4.139311139656056,44999.0,0.008600,3.004335,-12.668148,-2.024688,0.017977,2.033905,13.023745
18.89359354792485,44999.0,-0.043499,10.031000,-54.216931,-6.799410,-0.069868,6.672912,38.731440
5.472066192108001,44999.0,-0.260803,40.034246,-155.600356,-27.102140,0.081334,26.507046,168.057708
0.3299564440962265,44999.0,0.002206,0.997296,-4.103489,-0.675812,-0.003181,0.672637,4.535973
3.611043842811059,44999.0,0.050417,9.982634,-42.007679,-6.679235,0.011707,6.794519,39.439889
cluster,44999.0,0.497922,0.500001,0.000000,0.000000,0.000000,1.000000,1.000000


In [10]:
dataset.head()

,2318.4402090840676,0.9095661384526199,-4.139311139656056,18.89359354792485,5.472066192108001,0.3299564440962265,3.611043842811059,cluster
0,2369.017404,-1.754597,1.313440,3.161897,-34.433554,1.110685,-18.044512,0
1,2453.383691,1.088862,-1.227993,-9.131891,-11.780860,-0.086343,-9.914807,1
2,2433.591570,-5.839638,3.705717,1.830827,5.159389,0.467818,-20.351105,1
3,2459.126812,2.903708,1.819649,7.543625,27.984036,0.279439,12.624162,1
4,2389.078792,1.728807,6.556076,6.190018,-34.167965,-0.798856,-7.729004,0


In [11]:
dataset.shape

(44999, 8)

In [12]:
dataset.isnull().sum()

,0
2318.4402090840676,0
0.9095661384526199,0
-4.139311139656056,0
18.89359354792485,0
5.472066192108001,0
0.3299564440962265,0
3.611043842811059,0
cluster,0


In [13]:
X = dataset.drop(['cluster'], axis=1)
y = dataset['cluster']

In [14]:
from sklearn import svm
import psutil
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##SVM

In [16]:
svm_model = svm.SVC(kernel='rbf', C=1)

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

process = psutil.Process()
cpu_utilization = psutil.cpu_percent(interval=1)
memory_usage = process.memory_info().rss / 1024

print("SVM Model Evaluation")
print("--------------------")
print(f"CPU Utilization: {cpu_utilization}%")
print(f"Memory Usage: {memory_usage} KB")
accuracy = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", accuracy)
precision = precision_score(y_test, y_pred_svm)
print("SVM Precision:", precision)
recall = recall_score(y_test, y_pred_svm)
print("SVM Recall:", recall)
f1 = f1_score(y_test, y_pred_svm)
print("SVM F1 Score:", f1)

SVM Model Evaluation
--------------------
CPU Utilization: 1.5%
Memory Usage: 261672.0 KB
SVM Accuracy: 0.9986666666666667
SVM Precision: 0.9991051454138703
SVM Recall: 0.9982118909253465
SVM F1 Score: 0.9986583184257602


In [1]:
import sys
import pickle

In [4]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [17]:
with open('/content/drive/MyDrive/IoT-IDS-Shared/svm_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

model_size = sys.getsizeof(pickle.dumps(svm_model))
print(f"SVM Model Size: {model_size / 1024:.2f} KB")

SVM Model Size: 880.97 KB


##KNN

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from scipy.spatial.distance import mahalanobis
from sklearn.neighbors import KNeighborsClassifier

In [19]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [20]:
cov_matrix = np.cov(X_train.T)
inv_cov_matrix = np.linalg.inv(cov_matrix)

In [21]:
def mahalanobis_distance(a, b):
    return mahalanobis(a, b, inv_cov_matrix)

In [22]:
knn_model = KNeighborsClassifier(n_neighbors=2, metric='mahalanobis', metric_params={'V': inv_cov_matrix})

knn_model.fit(X_train, y_train)

y_pred_knn = knn_model.predict(X_test)

process = psutil.Process()
cpu_utilization = psutil.cpu_percent(interval=1)
memory_usage = process.memory_info().rss / 1024

print("KNN Model Evaluation")
print("--------------------")
print(f"CPU Utilization: {cpu_utilization}%")
print(f"Memory Usage: {memory_usage} KB")
accuracy = accuracy_score(y_test, y_pred_knn)
print("KNN Accuracy:", accuracy)
precision = precision_score(y_test, y_pred_knn)
print("KNN Precision:", precision)
recall = recall_score(y_test, y_pred_knn)
print("KNN Recall:", recall)
f1 = f1_score(y_test, y_pred_knn)
print("KNN F1 Score:", f1)

KNN Model Evaluation
--------------------
CPU Utilization: 2.0%
Memory Usage: 269368.0 KB
KNN Accuracy: 0.994
KNN Precision: 0.9984212900315742
KNN Recall: 0.9894948591864103
KNN F1 Score: 0.9939380332285586


In [23]:
with open('/content/drive/MyDrive/research_paper/knn_model.pkl', 'wb') as f:
    pickle.dump(knn_model, f)

model_size_knn = sys.getsizeof(pickle.dumps(knn_model))
print(f"KNN Model Size: {model_size_knn / 1024 :.2f} KB")

KNN Model Size: 2709.34 KB


##CNN

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

class NumericalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = NumericalDataset(X_train, y_train)
test_dataset = NumericalDataset(X_test, y_test)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool1d(1)
        self.fc1_input_size = self._get_fc1_input_size(torch.randn(1, 1, X_train.shape[1]))
        self.fc1 = nn.Linear(self.fc1_input_size, 128)
        self.fc2 = nn.Linear(128, 1)

    def _get_fc1_input_size(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.relu(self.conv3(x))
        x = self.pool3(x)
        return x.view(-1).shape[0]

    def forward(self, x):
        x = x.unsqueeze(1)
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        x = torch.relu(self.conv3(x))
        x = self.pool3(x)
        x = x.view(-1, self.fc1_input_size)
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return x

In [25]:
model = CNNModel()
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.BCELoss()

In [26]:
for epoch in range(20):
    model.train()
    total_loss = 0
    for batch in train_loader:
        X_batch, y_batch = batch
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

    model.eval()
    with torch.no_grad():
        total_correct = 0
        for batch in test_loader:
            X_batch, y_batch = batch
            outputs = model(X_batch)
            predictions = torch.round(outputs)
            total_correct += (predictions == y_batch.view(-1, 1)).sum().item()
        accuracy = total_correct / len(test_loader.dataset)
        print(f'Test Accuracy: {accuracy:.4f}')

Epoch 1, Loss: 0.07735921154451919
Test Accuracy: 0.9840
Epoch 2, Loss: 0.03607180249808718
Test Accuracy: 0.9808
Epoch 3, Loss: 0.027077574761524576
Test Accuracy: 0.9922
Epoch 4, Loss: 0.02320506527342756
Test Accuracy: 0.9913
Epoch 5, Loss: 0.023277438568175865
Test Accuracy: 0.9894
Epoch 6, Loss: 0.018567109173661948
Test Accuracy: 0.9922
Epoch 7, Loss: 0.019862884633063134
Test Accuracy: 0.9850
Epoch 8, Loss: 0.017647181029593166
Test Accuracy: 0.9907
Epoch 9, Loss: 0.017053623782852117
Test Accuracy: 0.9886
Epoch 10, Loss: 0.015333503272335906
Test Accuracy: 0.9927
Epoch 11, Loss: 0.014735404802942498
Test Accuracy: 0.9926
Epoch 12, Loss: 0.014371108068030124
Test Accuracy: 0.9946
Epoch 13, Loss: 0.013869441935842407
Test Accuracy: 0.9969
Epoch 14, Loss: 0.013106421442956524
Test Accuracy: 0.9944
Epoch 15, Loss: 0.013618334698127859
Test Accuracy: 0.9896
Epoch 16, Loss: 0.0109745592625001
Test Accuracy: 0.9906
Epoch 17, Loss: 0.011507450226610589
Test Accuracy: 0.9938
Epoch 18, L

In [27]:
y_pred = model(torch.tensor(X_test, dtype=torch.float32))
y_pred_class = torch.round(y_pred)

In [28]:
process = psutil.Process()
cpu_utilization = psutil.cpu_percent(interval=1)
memory_usage = process.memory_info().rss / 1024

print("CNN Model Evaluation")
print("--------------------")
print(f"CPU Utilization: {cpu_utilization}%")
print(f"Memory Usage: {memory_usage} KB")
print(f"Accuracy: {accuracy_score(y_test, y_pred_class.detach().numpy()):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_class.detach().numpy()):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_class.detach().numpy()):.4f}")
print(f"F1 Score: {f1_score(y_test, y_pred_class.detach().numpy()):.4f}")

CNN Model Evaluation
--------------------
CPU Utilization: 52.8%
Memory Usage: 1089332.0 KB
Accuracy: 0.9959
Precision: 0.9975
Recall: 0.9942
F1 Score: 0.9959


In [29]:
torch.save(model.state_dict(), '/content/drive/MyDrive/research_paper/cnn_model.pth')

# Calculate model size
model_size_cnn = os.path.getsize('/content/drive/MyDrive/research_paper/cnn_model.pth') / 1024  # in KB
print(f"Model Size: {model_size_cnn:.2f} KB")

Model Size: 190.35 KB
